# retraining_execution_handoff_current_v1

Purpose: fold the current retention audit, SOXLBacktest qlib reference, release16 failed-fold diagnostics, state feature mining, target-contract revision, and latest checkpoint replay into one executable handoff for the next retraining loop.

This card does not train, promote, or retain an actor. It defines the only allowed next training branch and the gates that must pass before a model can be kept.


## Experiment Governance

- Baselines: canonical `soxl_backtest_policy_reimplementation_v1` / `soxlbacktest_qlib_walkforward_baseline_v1`; direct comparison `release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1` and `retraining_retention_current_audit_v1`.
- Role: `diagnose` and training handoff. No promotion.
- Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`, then `ytd_2026` dashboard replay after any single retained actor candidate exists.
- Accepted trade-off: keep walk-forward MaxDD `<20%`; require relative return `>=30pp` on every walk-forward fold; require YTD return to clear both SOXLBacktest reference and current-best-alignment floors before actor retention.
- Changed layer: `model_selection`, `training_target`, `feature`, `reward`, and `selection` handoff only.
- External reference: SOXLBacktest qlib buy/sell reference baseline and official daily-path replay metrics.
- Hypothesis: the existing release16 checkpoint family fails because state representation and target transfer underparticipate in rebound/upside states, so the next training branch must consume the drawdown/upside target contract plus an auxiliary state head rather than rerun blind PPO or a post-hoc overlay.
- Acceptance: this handoff passes only if it identifies a non-duplicate, constrained next training branch and keeps `promotion_allowed=false` / `actor_retention_allowed=false` until future replay evidence passes all gates.
- Failure mode: if the latest checkpoint replay remains partial-pass and no train-only target contract exists, block training and return to target mining.
- Closest success: `wf_pass_ytd18_15_15_02` keeps walk-forward drawdown below the gate but lacks enough YTD return.
- Closest failure: `release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1` preserves risk on most folds but misses relative-return gates on `wf_2020` and `wf_2023_2024`.
- Duplicate check: this is not a renamed profile or threshold-only tweak; it consolidates current evidence into a constrained training contract and explicitly blocks retention.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
import json
from pathlib import Path
from typing import Any

from IPython.display import Markdown, display
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    for parent in Path.cwd().parents:
        if (parent / "AGENTS.md").exists():
            ROOT = parent
            break

EXPERIMENT = "retraining_execution_handoff_current_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

REQ_RELATIVE_PP = 30.0
REQ_MAX_DD_PCT = 20.0

SOURCE_CARDS = {
    "current_audit": ROOT / "runtime/model_selection/card/retraining_retention_current_audit_v1",
    "soxlbacktest_qlib": ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1",
    "remaining_failedfold": ROOT / "runtime/model_selection/card/release16_remaining_failedfold_diagnostic_v1",
    "state_feature_mining": ROOT / "runtime/model_selection/card/release16_failedfold_state_feature_mining_v1",
    "plain_target_recipe": ROOT / "runtime/model_selection/card/release16_rulebase_reversal_target_recipe_v1",
    "drawdown_upside_target": ROOT / "runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_target_v1",
    "drawdown_upside_replay": ROOT / "runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1",
}

def read_json(path: Path) -> dict[str, Any]:
    with path.open() as f:
        return json.load(f)

def write_json(path: Path, data: Any) -> None:
    with path.open("w") as f:
        json.dump(data, f, indent=2, sort_keys=True)
        f.write("\n")

def rel(path: Path) -> str:
    return path.relative_to(ROOT).as_posix()

summaries = {}
for name, source_card in SOURCE_CARDS.items():
    summary_path = source_card / "summary.json"
    if summary_path.exists():
        summaries[name] = read_json(summary_path)
    else:
        summaries[name] = {"missing_summary": True, "card": rel(source_card)}

created_at = datetime.now(timezone.utc).isoformat()
print({"experiment": EXPERIMENT, "created_at": created_at, "sources": list(SOURCE_CARDS)})


{'experiment': 'retraining_execution_handoff_current_v1', 'created_at': '2026-06-14T17:54:36.428537+00:00', 'sources': ['current_audit', 'soxlbacktest_qlib', 'remaining_failedfold', 'state_feature_mining', 'plain_target_recipe', 'drawdown_upside_target', 'drawdown_upside_replay']}


In [2]:
# Build the evidence chain in dependency order.
evidence_rows = [
    {
        "step": 1,
        "card_key": "soxlbacktest_qlib",
        "card": rel(SOURCE_CARDS["soxlbacktest_qlib"]),
        "decision": summaries["soxlbacktest_qlib"].get("decision"),
        "role": "reference_baseline",
        "handoff_signal": "qlib-aligned buy/sell timing reference; not promotable by itself",
    },
    {
        "step": 2,
        "card_key": "current_audit",
        "card": rel(SOURCE_CARDS["current_audit"]),
        "decision": summaries["current_audit"].get("decision"),
        "role": "current_retention_gate",
        "handoff_signal": "no existing replayable model passes retention; no runnable RL actor in dashboard",
    },
    {
        "step": 3,
        "card_key": "drawdown_upside_replay",
        "card": rel(SOURCE_CARDS["drawdown_upside_replay"]),
        "decision": summaries["drawdown_upside_replay"].get("decision"),
        "role": "frontier_checkpoint_replay",
        "handoff_signal": "partial pass only; failed folds must drive next training objective",
    },
    {
        "step": 4,
        "card_key": "remaining_failedfold",
        "card": rel(SOURCE_CARDS["remaining_failedfold"]),
        "decision": summaries["remaining_failedfold"].get("decision"),
        "role": "failure_attribution",
        "handoff_signal": "state transfer and representation gap; low-position up days dominate gap",
    },
    {
        "step": 5,
        "card_key": "state_feature_mining",
        "card": rel(SOURCE_CARDS["state_feature_mining"]),
        "decision": summaries["state_feature_mining"].get("decision"),
        "role": "feature_handoff",
        "handoff_signal": "rulebase_reversal feature set is the best transfer candidate",
    },
    {
        "step": 6,
        "card_key": "drawdown_upside_target",
        "card": rel(SOURCE_CARDS["drawdown_upside_target"]),
        "decision": summaries["drawdown_upside_target"].get("decision"),
        "role": "train_only_target_contract",
        "handoff_signal": "target contract is train-only and passed leakage/scope checks",
    },
]

evidence_chain = pd.DataFrame(evidence_rows)
evidence_chain.to_csv(CARD / "evidence_chain.csv", index=False)
display(evidence_chain)


,step,card_key,card,decision,role,handoff_signal
0,1,soxlbacktest_qlib,runtime/model_selection/card/soxlbacktest_qlib...,soxlbacktest_reference_qlib_aligned_not_promot...,reference_baseline,qlib-aligned buy/sell timing reference; not pr...
1,2,current_audit,runtime/model_selection/card/retraining_retent...,diagnose_release16_failed_folds_before_full_re...,current_retention_gate,no existing replayable model passes retention;...
2,3,drawdown_upside_replay,runtime/model_selection/card/release16_rulebas...,release16_rulebase_reversal_drawdown_upside_ch...,frontier_checkpoint_replay,partial pass only; failed folds must drive nex...
3,4,remaining_failedfold,runtime/model_selection/card/release16_remaini...,release16_overlay_failure_is_state_transfer_an...,failure_attribution,state transfer and representation gap; low-pos...
4,5,state_feature_mining,runtime/model_selection/card/release16_failedf...,failedfold_state_head_features_ready_for_targe...,feature_handoff,rulebase_reversal feature set is the best tran...
5,6,drawdown_upside_target,runtime/model_selection/card/release16_rulebas...,release16_rulebase_reversal_drawdown_upside_ta...,train_only_target_contract,target contract is train-only and passed leaka...


In [3]:
# Quantify the latest replay gaps and attach failed-fold attribution.
replay = pd.read_csv(SOURCE_CARDS["drawdown_upside_replay"] / "checkpoint_selection_by_fold.csv")
missed = pd.read_csv(SOURCE_CARDS["remaining_failedfold"] / "missed_up_days_summary.csv")
bc_repr = pd.read_csv(SOURCE_CARDS["remaining_failedfold"] / "bc_representation_summary.csv")
target_scope = pd.read_csv(SOURCE_CARDS["drawdown_upside_target"] / "target_scope_audit.csv")

fold_gap = replay.copy()
fold_gap["required_relative_return_pp"] = REQ_RELATIVE_PP
fold_gap["required_max_dd_pct"] = REQ_MAX_DD_PCT
fold_gap["relative_gap_to_gate_pp"] = (REQ_RELATIVE_PP - fold_gap["selected_relative_return"]).clip(lower=0)
fold_gap["maxdd_gap_to_gate_pct"] = (fold_gap["selected_max_dd"] - REQ_MAX_DD_PCT).clip(lower=0)
fold_gap["gate_pass"] = (fold_gap["selected_relative_return"] >= REQ_RELATIVE_PP) & (fold_gap["selected_max_dd"] < REQ_MAX_DD_PCT)
fold_gap = fold_gap.merge(missed, on="fold", how="left")
fold_gap = fold_gap.merge(
    bc_repr[[
        "fold",
        "bc_all_accuracy",
        "predicted_cash_heavier_than_label",
        "target_share_ge50",
        "selected_share_ge50",
        "target_mean_pos",
        "selected_mean_pos",
    ]],
    on="fold",
    how="left",
)
fold_gap = fold_gap.merge(
    target_scope[["fold", "changed_rows_train", "changed_rows_valid", "changed_rows_test", "target_contract_gate"]],
    on="fold",
    how="left",
)

def target_action(row: pd.Series) -> str:
    if bool(row.get("gate_pass")):
        return "guard_only_keep_target_unchanged"
    if row["fold"] == "wf_2020":
        return "repair_drawdown_cap_without_suppressing_reentry; add state-head reward for rebound participation"
    if row["fold"] == "wf_2023_2024":
        return "raise sustained-upside participation transfer; add trend/reversal state-head reward and checkpoint penalty"
    return "diagnose_before_training"

fold_gap["required_training_action"] = fold_gap.apply(target_action, axis=1)
selected_cols = [
    "fold",
    "selected_candidate",
    "selected_relative_return",
    "selected_max_dd",
    "selected_return",
    "selected_avg_pos",
    "gate_pass",
    "relative_gap_to_gate_pp",
    "maxdd_gap_to_gate_pct",
    "days",
    "mean_selected_prev_position",
    "sum_soxl_up_day_return_pp",
    "sum_incremental_if_50pct_pp",
    "bc_all_accuracy",
    "predicted_cash_heavier_than_label",
    "target_share_ge50",
    "selected_share_ge50",
    "target_mean_pos",
    "selected_mean_pos",
    "changed_rows_train",
    "changed_rows_valid",
    "changed_rows_test",
    "target_contract_gate",
    "required_training_action",
]
fold_gap_out = fold_gap[selected_cols]
fold_gap_out.to_csv(CARD / "fold_gap_to_training_targets.csv", index=False)
display(fold_gap_out)


,fold,selected_candidate,selected_relative_return,selected_max_dd,selected_return,selected_avg_pos,gate_pass,relative_gap_to_gate_pp,maxdd_gap_to_gate_pct,days,...,predicted_cash_heavier_than_label,target_share_ge50,selected_share_ge50,target_mean_pos,selected_mean_pos,changed_rows_train,changed_rows_valid,changed_rows_test,target_contract_gate,required_training_action
0,wf_2020,bc_initialization,-0.427053,19.326906,57.666518,0.302925,False,30.427053,0.0,47.0,...,True,0.301623,0.071429,0.364056,0.230203,92,0,0,True,repair_drawdown_cap_without_suppressing_reentr...
1,wf_2022,bc_initialization,71.714980,15.617043,-14.873092,0.081384,True,0.000000,0.0,NaN,...,False,0.090909,0.000000,0.200429,0.081384,0,0,0,True,guard_only_keep_target_unchanged
2,wf_2023_2024,bc_initialization,2.991718,13.624617,2.635083,0.216822,False,27.008282,0.0,66.0,...,True,0.300265,0.000000,0.236502,0.083241,151,0,0,True,raise sustained-upside participation transfer;...


In [4]:
# Feature and target contract handoff.
feature_scores = pd.read_csv(SOURCE_CARDS["state_feature_mining"] / "test_state_feature_set_scores.csv")
train_feature_scores = pd.read_csv(SOURCE_CARDS["state_feature_mining"] / "train_overlay_feature_set_scores.csv")
alignment = pd.read_csv(SOURCE_CARDS["state_feature_mining"] / "state_transfer_feature_alignment.csv")
plain_recipe_scores = pd.read_csv(SOURCE_CARDS["plain_target_recipe"] / "recipe_scores.csv")

feature_handoff = (
    feature_scores.merge(
        train_feature_scores[["feature_set", "holdout_auc_min", "holdout_accuracy_min", "representation_gate"]].rename(
            columns={
                "holdout_auc_min": "train_overlay_auc_min",
                "holdout_accuracy_min": "train_overlay_accuracy_min",
                "representation_gate": "train_overlay_representation_gate",
            }
        ),
        on="feature_set",
        how="left",
    )
    .merge(
        plain_recipe_scores[["feature_set", "threshold", "state_precision", "state_recall", "effective_false_positive_rate", "recipe_gate", "recipe_loss"]],
        on="feature_set",
        how="left",
    )
)
feature_handoff["preferred_for_next_training"] = feature_handoff["feature_set"].eq("rulebase_reversal")
feature_handoff = feature_handoff.sort_values(
    ["preferred_for_next_training", "representation_gate", "train_overlay_representation_gate", "holdout_auc_min"],
    ascending=[False, False, False, False],
)
feature_handoff.to_csv(CARD / "feature_handoff.csv", index=False)

top_aligned = (
    alignment[alignment["effect_sign_aligned"].astype(bool)]
    .sort_values(["joint_abs_effect", "effect_abs_gap"], ascending=[False, True])
    .head(12)
)
top_aligned.to_csv(CARD / "top_state_transfer_features.csv", index=False)

display(feature_handoff[[
    "feature_set",
    "preferred_for_next_training",
    "holdout_auc_min",
    "holdout_accuracy_min",
    "train_overlay_auc_min",
    "train_overlay_accuracy_min",
    "threshold",
    "state_precision",
    "state_recall",
    "effective_false_positive_rate",
    "recipe_gate",
]])
display(top_aligned[["feature_set", "feature", "test_effect_size", "train_effect_size", "joint_abs_effect", "effect_abs_gap"]])


,feature_set,preferred_for_next_training,holdout_auc_min,holdout_accuracy_min,train_overlay_auc_min,train_overlay_accuracy_min,threshold,state_precision,state_recall,effective_false_positive_rate,recipe_gate
0,rulebase_reversal,True,0.863622,0.630952,0.666731,0.576004,0.5,0.437356,0.683894,0.0,True
1,rulebase_reversal,True,0.863622,0.630952,0.666731,0.576004,0.0,0.422208,0.808894,0.0,False
2,rulebase_reversal,True,0.863622,0.630952,0.666731,0.576004,-0.5,0.385517,0.870192,0.0,False
3,rulebase_reversal,True,0.863622,0.630952,0.666731,0.576004,1.0,0.426230,0.468750,0.0,False
4,rulebase_reversal,True,0.863622,0.630952,0.666731,0.576004,-1.0,0.333765,0.930288,0.0,False
5,rulebase_reversal,True,0.863622,0.630952,0.666731,0.576004,-1.5,0.302711,0.966346,0.0,False
6,rulebase_reversal,True,0.863622,0.630952,0.666731,0.576004,1.5,0.412903,0.307692,0.0,False
7,rulebase_reversal,True,0.863622,0.630952,0.666731,0.576004,2.0,0.439716,0.149038,0.0,False
8,transition_recovery,False,0.603425,0.678571,0.608788,0.495264,NaN,NaN,NaN,NaN,NaN
9,compact_state_head,False,0.690711,0.539683,0.661843,0.579612,0.5,0.459175,0.628606,0.0,False


,feature_set,feature,test_effect_size,train_effect_size,joint_abs_effect,effect_abs_gap
0,rulebase_reversal,RB_SELL_COMPOSITE,0.489719,0.711590,0.489719,0.221871
1,compact_state_head,mom_5,0.467311,0.407630,0.407630,0.059682
2,momentum_drawdown,mom_5,0.467311,0.407630,0.407630,0.059682
3,rulebase_reversal,RB_RSI6_OVERSOLD,-0.400469,-0.359886,0.359886,0.040583
4,compact_state_head,RB_TREND_REVERSAL,0.357359,0.678399,0.357359,0.321041
5,rulebase_reversal,RB_TREND_REVERSAL,0.357359,0.678399,0.357359,0.321041
6,rulebase_reversal,RB_CUMULATIVE_DROP_SEVERITY,0.328998,0.713118,0.328998,0.384120
7,momentum_drawdown,anchor_gap_180,0.325367,0.696342,0.325367,0.370975
8,transition_recovery,TRANS_SPRING_COMPRESSION,-0.330532,-0.292406,0.292406,0.038126
9,rulebase_reversal,RB_PROFIT_TAKE_PROXY,0.269991,0.781418,0.269991,0.511427


In [5]:
current_audit = summaries["current_audit"]
soxl_ref = current_audit.get("soxlbacktest_qlib_reference", {})
retention_gate = current_audit.get("current_retention_gate", {})
drawdown_target_summary = summaries["drawdown_upside_target"]

failed_folds = fold_gap_out.loc[~fold_gap_out["gate_pass"], "fold"].tolist()
all_target_scope_clean = bool(target_scope["target_contract_gate"].all() and (target_scope["changed_rows_valid"].sum() == 0) and (target_scope["changed_rows_test"].sum() == 0))
replay_partial_pass = len(failed_folds) > 0
feature_recipe_ready = bool(feature_handoff.loc[feature_handoff["feature_set"].eq("rulebase_reversal"), "recipe_gate"].fillna(False).any())
target_contract_ready = bool(drawdown_target_summary.get("target_contract_gate", False) and all_target_scope_clean)

training_branch_allowed = bool(feature_recipe_ready and target_contract_ready and replay_partial_pass)

contract = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "decision": "constrained_release16_state_head_training_handoff_no_promotion" if training_branch_allowed else "training_handoff_blocked_missing_contract_or_features",
    "training_branch_allowed": training_branch_allowed,
    "allowed_training_branch": "release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1" if training_branch_allowed else None,
    "same_checkpoint_or_posthoc_overlay_rerun_allowed": False,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "dashboard_runnable_model_expected_now": False,
    "failed_walkforward_folds": failed_folds,
    "walkforward_gate": {
        "required_relative_return_pp": REQ_RELATIVE_PP,
        "required_max_drawdown_pct_lt": REQ_MAX_DD_PCT,
        "basis": "official_daily_path_rebased_eval_window",
    },
    "current_retention_gate": {
        "profiles_scanned": retention_gate.get("profiles_scanned"),
        "passing_profile_count": retention_gate.get("passing_profile_count"),
        "current_best_return_pct": retention_gate.get("current_best_return_pct"),
        "required_best_return_pct": retention_gate.get("required_best_return_pct"),
        "reference_return_pct": retention_gate.get("reference_return_pct"),
    },
    "soxlbacktest_qlib_reference": soxl_ref,
    "source_target_contract": {
        "card": rel(SOURCE_CARDS["drawdown_upside_target"]),
        "path": rel(SOURCE_CARDS["drawdown_upside_target"] / "drawdown_upside_bc_target_contract.csv"),
        "target_contract_ready": target_contract_ready,
        "scope_clean": all_target_scope_clean,
        "wf_2020_changed_train_rows": int(target_scope.loc[target_scope["fold"].eq("wf_2020"), "changed_rows_train"].fillna(0).sum()),
        "wf_2023_2024_changed_train_rows": int(target_scope.loc[target_scope["fold"].eq("wf_2023_2024"), "changed_rows_train"].fillna(0).sum()),
        "changed_eval_or_test_rows": int(target_scope["changed_rows_valid"].sum() + target_scope["changed_rows_test"].sum()),
    },
    "required_model_changes": [
        "Consume drawdown_upside_bc_target_contract.csv as train-only supervision, not eval labels.",
        "Add an auxiliary rulebase_reversal state head or equivalent representation loss using the top state-transfer features.",
        "Use official daily-path replay as the checkpoint-selection metric; ignore raw metrics when they disagree.",
        "Penalize underparticipation on low-position up-day states in wf_2020 and wf_2023_2024 while keeping MaxDD <20%.",
        "Export replayable model outputs and rebuild dashboard_data/v2 before any retention review.",
    ],
    "hard_blocks": [
        "Do not retain a checkpoint from any fold-local diagnostic actor.",
        "Do not promote evidence-only dashboard profiles as runnable models.",
        "Do not rerun the same release16 drawdown/upside checkpoint replay and call it progress; it already partial-passed only.",
        "Do not relax SOXLBacktest qlib reference scoring or the current-best YTD return floor.",
    ],
    "validation_after_training": [
        "fixed_split replay with dashboard-visible position/price/OHLC output",
        "wf_2020 official daily-path relative return >=30pp and MaxDD <20%",
        "wf_2022 official daily-path relative return >=30pp and MaxDD <20%",
        "wf_2023_2024 official daily-path relative return >=30pp and MaxDD <20%",
        "YTD return >= SOXLBacktest qlib reference and >= current-best alignment floor",
        "model_replay_freshness current/refreshed count >0 for the retained profile",
        "dashboard v2 Run source lists the retained profile as runnable",
    ],
    "evidence_sidecars": {
        "evidence_chain": rel(CARD / "evidence_chain.csv"),
        "fold_gap_to_training_targets": rel(CARD / "fold_gap_to_training_targets.csv"),
        "feature_handoff": rel(CARD / "feature_handoff.csv"),
        "top_state_transfer_features": rel(CARD / "top_state_transfer_features.csv"),
    },
}

write_json(CARD / "current_training_contract.json", contract)
contract


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_85042/1426490115.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  feature_recipe_ready = bool(feature_handoff.loc[feature_handoff["feature_set"].eq("rulebase_reversal"), "recipe_gate"].fillna(False).any())


{'experiment': 'retraining_execution_handoff_current_v1',
 'created_at': '2026-06-14T17:54:36.428537+00:00',
 'decision': 'constrained_release16_state_head_training_handoff_no_promotion',
 'training_branch_allowed': True,
 'allowed_training_branch': 'release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1',
 'same_checkpoint_or_posthoc_overlay_rerun_allowed': False,
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'dashboard_runnable_model_expected_now': False,
 'failed_walkforward_folds': ['wf_2020', 'wf_2023_2024'],
 'walkforward_gate': {'required_relative_return_pp': 30.0,
  'required_max_drawdown_pct_lt': 20.0,
  'basis': 'official_daily_path_rebased_eval_window'},
 'current_retention_gate': {'profiles_scanned': 13,
  'passing_profile_count': 0,
  'current_best_return_pct': 224.361539,
  'required_best_return_pct': 213.143462,
  'reference_return_pct': 132.077978},
 'soxlbacktest_qlib_reference': {'all_windows_match': True,
  'daily_end': '2026-06-09',

In [6]:
summary = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "decision": contract["decision"],
    "training_branch_allowed": contract["training_branch_allowed"],
    "allowed_training_branch": contract["allowed_training_branch"],
    "promotion_allowed": contract["promotion_allowed"],
    "actor_retention_allowed": contract["actor_retention_allowed"],
    "failed_walkforward_folds": contract["failed_walkforward_folds"],
    "source_target_contract": contract["source_target_contract"],
    "current_retention_gate": contract["current_retention_gate"],
    "soxlbacktest_qlib_reference": contract["soxlbacktest_qlib_reference"],
    "required_model_changes": contract["required_model_changes"],
    "hard_blocks": contract["hard_blocks"],
    "validation_after_training": contract["validation_after_training"],
    "sidecars": [
        "evidence_chain.csv",
        "fold_gap_to_training_targets.csv",
        "feature_handoff.csv",
        "top_state_transfer_features.csv",
        "current_training_contract.json",
        "summary.json",
        "conclusion.md",
    ],
    "conclusion": (
        "Open a constrained release16 rulebase-reversal drawdown/upside state-head training probe only; "
        "do not promote or retain any actor until future fixed-split, walk-forward, YTD, SOXLBacktest-reference, "
        "model-replay, and dashboard runnable-source gates all pass."
    ),
}
write_json(CARD / "summary.json", summary)

failed_fold_lines = []
for row in fold_gap_out.to_dict("records"):
    failed_fold_lines.append(
        f"- {row['fold']}: rel {row['selected_relative_return']:.2f}pp, MaxDD {row['selected_max_dd']:.2f}%, "
        f"gap {row['relative_gap_to_gate_pp']:.2f}pp, action `{row['required_training_action']}`."
    )

feature_lines = []
for row in top_aligned.head(6).to_dict("records"):
    feature_lines.append(
        f"- {row['feature_set']}/{row['feature']}: test effect {row['test_effect_size']:.3f}, "
        f"train effect {row['train_effect_size']:.3f}."
    )

lines = [
    "# Retraining Execution Handoff Current",
    "",
    f"Decision: `{summary['decision']}`.",
    "",
    f"Training branch allowed: `{summary['training_branch_allowed']}` for `{summary['allowed_training_branch']}`. This is an actor-probe handoff only; promotion and actor retention remain `False`.",
    "",
    "Latest replay gaps:",
    *failed_fold_lines,
    "",
    f"Required source target: `{contract['source_target_contract']['path']}`; scope clean `{contract['source_target_contract']['scope_clean']}`, changed eval/test rows `{contract['source_target_contract']['changed_eval_or_test_rows']}`.",
    "",
    "Primary feature handoff:",
    *feature_lines,
    "",
    "Hard gates after any training:",
    "- fixed split plus `wf_2020`, `wf_2022`, `wf_2023_2024` on official daily-path metrics.",
    "- every walk-forward fold relative return >=30pp and MaxDD <20%.",
    "- YTD return must clear SOXLBacktest qlib reference and current-best-alignment floors.",
    "- replay export must make the profile runnable in dashboard v2 before retention review.",
    "",
    "Risk note: do not rerun the same checkpoint/post-hoc overlay and call it a retained model. The next branch must add the state-head / representation objective and still pass full replay gates.",
    "",
    "Artifacts: `evidence_chain.csv`, `fold_gap_to_training_targets.csv`, `feature_handoff.csv`, `top_state_transfer_features.csv`, `current_training_contract.json`, `summary.json`.",
]
conclusion_md = "\n".join(lines) + "\n"
(CARD / "conclusion.md").write_text(conclusion_md)

display(Markdown(conclusion_md))


# Retraining Execution Handoff Current

Decision: `constrained_release16_state_head_training_handoff_no_promotion`.

Training branch allowed: `True` for `release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1`. This is an actor-probe handoff only; promotion and actor retention remain `False`.

Latest replay gaps:
- wf_2020: rel -0.43pp, MaxDD 19.33%, gap 30.43pp, action `repair_drawdown_cap_without_suppressing_reentry; add state-head reward for rebound participation`.
- wf_2022: rel 71.71pp, MaxDD 15.62%, gap 0.00pp, action `guard_only_keep_target_unchanged`.
- wf_2023_2024: rel 2.99pp, MaxDD 13.62%, gap 27.01pp, action `raise sustained-upside participation transfer; add trend/reversal state-head reward and checkpoint penalty`.

Required source target: `runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_target_v1/drawdown_upside_bc_target_contract.csv`; scope clean `True`, changed eval/test rows `0`.

Primary feature handoff:
- rulebase_reversal/RB_SELL_COMPOSITE: test effect 0.490, train effect 0.712.
- compact_state_head/mom_5: test effect 0.467, train effect 0.408.
- momentum_drawdown/mom_5: test effect 0.467, train effect 0.408.
- rulebase_reversal/RB_RSI6_OVERSOLD: test effect -0.400, train effect -0.360.
- compact_state_head/RB_TREND_REVERSAL: test effect 0.357, train effect 0.678.
- rulebase_reversal/RB_TREND_REVERSAL: test effect 0.357, train effect 0.678.

Hard gates after any training:
- fixed split plus `wf_2020`, `wf_2022`, `wf_2023_2024` on official daily-path metrics.
- every walk-forward fold relative return >=30pp and MaxDD <20%.
- YTD return must clear SOXLBacktest qlib reference and current-best-alignment floors.
- replay export must make the profile runnable in dashboard v2 before retention review.

Risk note: do not rerun the same checkpoint/post-hoc overlay and call it a retained model. The next branch must add the state-head / representation objective and still pass full replay gates.

Artifacts: `evidence_chain.csv`, `fold_gap_to_training_targets.csv`, `feature_handoff.csv`, `top_state_transfer_features.csv`, `current_training_contract.json`, `summary.json`.
